# 4b. Parameter Dynamics Across Learning

Are model parameters genuinely dynamic, or can a single static
parameter set explain behaviour across all learning phases?

If we claim PPC necessity is dynamic because the animal's internal
model is dynamic, we need to show the model parameters actually change.

Three levels of analysis, with increasing assumptions:

1. **Static vs dynamic** — does allowing parameters to vary across
   phases improve fit quality?
2. **Phase-blocked fitting** — fit parameters per phase
   (naive / expert / early post / late post)
3. **Per-session fitting** — noisy but assumption-free trajectory
4. **GP-linked SBI** — smooth trajectory with uncertainty (depends on 2c)

**Depends on:** 3a/3c (model assignments), 4a (shift detection)

In [1]:
%matplotlib inline
from shared_setup import *
from pathlib import Path

from analysis.adaptation import (
    detect_all_manipulations,
    classify_shift_type,
    build_phase_blocks,
)
from analysis.grid_search import (
    grid_search_cv, DEFAULT_GRID, _sessions_to_arrays,
)
# These are from the extensions module — merge into grid_search.py
from analysis.grid_search import (
    fit_sessions_blocked,
    fit_sessions_individual,
    compare_static_vs_dynamic,
)

## 1. Configuration

In [2]:
# Phases: how to block sessions for the phase-blocked fit
# These are relative to the first shift detected per animal
PHASE_DEFINITIONS = {
    'naive': {'type': 'first_n', 'n': 5},
    'expert': {'type': 'last_n_before_shift', 'n': 5},
    'early_post': {'type': 'first_n_after_shift', 'n': 5},
    'late_post': {'type': 'remaining_after_shift', 'skip': 5},
}

GRID_SEEDS = 5
BURN_IN = 1000

## 2. Load Data and Model Assignments

In [3]:
experiment, load_info = load_data()

# ── Load model assignments (from 3c or 3a) ──────────────────────────────────
assignment_map = {}

assignments_path = Path('../results/model_assignments.csv')
gs_anova_path = Path('../results/cv/cv_comparison_anova.csv')

if assignments_path.exists():
    assignments = pd.read_csv(assignments_path)
    assignment_map = dict(zip(assignments['animal_id'], assignments['winner']))
    print(f'Model assignments from consensus: {len(assignment_map)} animals')
elif gs_anova_path.exists():
    gs_df = pd.read_csv(gs_anova_path)
    if 'winner' in gs_df.columns:
        assignment_map = dict(zip(gs_df['animal_id'], gs_df['winner']))
        print(f'Model assignments from grid-search ANOVA: {len(assignment_map)} animals')
else:
    print('No model assignments found — defaulting all animals to BE.')
    print('Run notebooks 3a and 3c first for proper assignments.')

if assignment_map:
    for aid, model in sorted(assignment_map.items()):
        print(f'  {aid}: {model}')

Loaded 12 animals, 433 total sessions
Loaded 12 animals, 433 sessions from CSV
Model assignments from grid-search ANOVA: 2 animals
  SS01: Inconclusive
  SS04: Inconclusive


## 3. Build Phase Blocks Per Animal

In [4]:
# ── Build for all animals ───────────────────────────────────────────────────
animal_phases = {}

for aid in experiment.animal_ids:
    animal = experiment.get_animal(aid)
    sessions = animal.get_sessions(stage=STAGE)
    if len(sessions) < 10:
        continue

    blocks = build_phase_blocks(animal)
    assigned = assignment_map.get(aid, None)  # from 3a/3c, may be None or 'Inconclusive'

    animal_phases[aid] = {
        'animal': animal,
        'sessions': sessions,
        'blocks': blocks,
        'assigned_model': assigned,  # reference only — we fit both models
    }
    phase_summary = {k: len(v) for k, v in blocks.items()}
    assigned_str = assigned if assigned in ('BE', 'SC') else f'{assigned} (inconclusive)'
    print(f'{aid} (assigned: {assigned_str}): {phase_summary}')

print(f'\n{len(animal_phases)} animals with sufficient sessions')

SS01 (assigned: Inconclusive (inconclusive)): {'naive': 5, 'expert': 5, 'early_post': 5}
SS04 (assigned: Inconclusive (inconclusive)): {'naive': 5, 'expert': 5, 'early_post': 5}
SS05 (assigned: None (inconclusive)): {'naive': 5, 'expert': 5, 'early_post': 5}
SS06 (assigned: None (inconclusive)): {'naive': 5, 'expert': 5, 'early_post': 5}
SS07 (assigned: None (inconclusive)): {'naive': 5, 'expert': 5, 'early_post': 5}
SS08 (assigned: None (inconclusive)): {'naive': 5, 'expert': 5, 'early_post': 5}
SS09 (assigned: None (inconclusive)): {'naive': 5, 'expert': 5, 'early_post': 5}
SS10 (assigned: None (inconclusive)): {'naive': 5, 'expert': 5}
SS11 (assigned: None (inconclusive)): {'naive': 5, 'expert': 5, 'early_post': 4}
SS12 (assigned: None (inconclusive)): {'naive': 5, 'expert': 5}
SS13 (assigned: None (inconclusive)): {'naive': 5, 'expert': 5, 'early_post': 5}

11 animals with sufficient sessions


## 4. Static vs Dynamic: Is a Dynamic Model Necessary?

Fit static parameters (all sessions pooled) and compare against
phase-blocked fits. If the phase-blocked fit isn't meaningfully
better, parameters aren't changing.

In [ ]:
comparison_results = {}  # aid -> {'BE': result, 'SC': result}

for aid, info in animal_phases.items():
    assigned = info['assigned_model']
    print(f'\n{aid} (assigned: {assigned})...')

    animal_comp = {}
    for mt in ['BE', 'SC']:
        try:
            result = compare_static_vs_dynamic(
                sessions=info['sessions'],
                model_type=mt,
                phase_blocks=info['blocks'],
                burn_in=BURN_IN,
                seed=42,
                n_seeds=GRID_SEEDS,
            )
            animal_comp[mt] = result
            marker = ' ←' if mt == assigned else ''
            print(f'  {mt}: static={result["static_total_error"]:.4f} '
                  f'dynamic={result["dynamic_total_error"]:.4f} '
                  f'ratio={result["improvement_ratio"]:.3f}{marker}')
        except Exception as e:
            print(f'  {mt}: FAILED ({e})')
            animal_comp[mt] = None

    comparison_results[aid] = animal_comp


SS01 (assigned: Inconclusive)...
  BE: FAILED (too many values to unpack (expected 4))
  SC: FAILED (too many values to unpack (expected 4))

SS04 (assigned: Inconclusive)...
  BE: FAILED (too many values to unpack (expected 4))
  SC: FAILED (too many values to unpack (expected 4))

SS05 (assigned: None)...
  BE: FAILED (too many values to unpack (expected 4))
  SC: FAILED (too many values to unpack (expected 4))

SS06 (assigned: None)...
  BE: FAILED (too many values to unpack (expected 4))
  SC: FAILED (too many values to unpack (expected 4))

SS07 (assigned: None)...
  BE: FAILED (too many values to unpack (expected 4))
  SC: FAILED (too many values to unpack (expected 4))

SS08 (assigned: None)...
  BE: FAILED (too many values to unpack (expected 4))
  SC: FAILED (too many values to unpack (expected 4))

SS09 (assigned: None)...
  BE: FAILED (too many values to unpack (expected 4))
  SC: FAILED (too many values to unpack (expected 4))

SS10 (assigned: None)...
  BE: FAILED (too ma

In [6]:
# ── Summary table ─────────────────────────────────────────────────────────────
summary_rows = []
for aid, comp in comparison_results.items():
    assigned = animal_phases[aid]['assigned_model']
    for mt in ['BE', 'SC']:
        r = comp.get(mt)
        if r is None:
            continue
        summary_rows.append({
            'animal_id': aid,
            'model': mt,
            'assigned': mt == assigned,
            'static_err': r['static_total_error'],
            'dynamic_err': r['dynamic_total_error'],
            'ratio': r['improvement_ratio'],
        })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False, float_format='%.4f'))

# Which model benefits more from dynamic fitting?
print('\n=== Dynamic vs Static: which model benefits more? ===')
for aid in comparison_results:
    comp = comparison_results[aid]
    for mt in ['BE', 'SC']:
        r = comp.get(mt)
        if r and r['improvement_ratio'] < 0.9:
            marker = ' ← assigned' if mt == animal_phases[aid]['assigned_model'] else ''
            print(f'  {aid} {mt}: dynamic is better (ratio={r["improvement_ratio"]:.3f}){marker}')

Empty DataFrame
Columns: []
Index: []

=== Dynamic vs Static: which model benefits more? ===


## 5. Phase-Blocked Fitting

For animals where the dynamic model is better, extract per-phase
parameters. The key question: does η (learning rate) differ across
naive → expert → early post → late post?

In [7]:
phase_fit_results = {}  # aid -> {'BE': result, 'SC': result}

for aid, info in animal_phases.items():
    print(f'{aid}...', end=' ')
    animal_fits = {}
    for mt in ['BE', 'SC']:
        try:
            result = fit_sessions_blocked(
                phase_blocks=info['blocks'],
                model_type=mt,
                burn_in=BURN_IN,
                seed=42,
                n_seeds=GRID_SEEDS,
            )
            animal_fits[mt] = result
        except Exception as e:
            print(f'{mt} failed ({e})', end=' ')
            animal_fits[mt] = None
    phase_fit_results[aid] = animal_fits
    print('done')

SS01... BE failed (too many values to unpack (expected 4)) SC failed (too many values to unpack (expected 4)) done
SS04... BE failed (too many values to unpack (expected 4)) SC failed (too many values to unpack (expected 4)) done
SS05... BE failed (too many values to unpack (expected 4)) SC failed (too many values to unpack (expected 4)) done
SS06... BE failed (too many values to unpack (expected 4)) SC failed (too many values to unpack (expected 4)) done
SS07... BE failed (too many values to unpack (expected 4)) SC failed (too many values to unpack (expected 4)) done
SS08... BE failed (too many values to unpack (expected 4)) SC failed (too many values to unpack (expected 4)) done
SS09... BE failed (too many values to unpack (expected 4)) SC failed (too many values to unpack (expected 4)) done
SS10... BE failed (too many values to unpack (expected 4)) SC failed (too many values to unpack (expected 4)) done
SS11... BE failed (too many values to unpack (expected 4)) SC failed (too many v

In [8]:
# ── Plot parameter values per phase ─────────────────────────────────────────
BE_PARAMS = ['sigma_percep', 'A_repulsion', 'eta_learning', 'eta_relax']
SC_PARAMS = ['sigma_percep', 'A_repulsion', 'gamma', 'sigma_update']

for aid, fits in phase_fit_results.items():
    assigned = animal_phases[aid]['assigned_model']

    for mt in ['BE', 'SC']:
        result = fits.get(mt)
        if result is None:
            continue

        param_names = BE_PARAMS if mt == 'BE' else SC_PARAMS
        phases = [p for p in result if result[p] is not None]
        if not phases:
            continue

        n_params = len(param_names)
        fig, axes = plt.subplots(1, n_params, figsize=(4 * n_params, 3.5))
        if n_params == 1:
            axes = [axes]

        for ax, pname in zip(axes, param_names):
            vals = []
            labels = []
            for phase in phases:
                p = result[phase].get('best_params', {})
                if p and pname in p:
                    vals.append(p[pname])
                    labels.append(phase)
            if vals:
                ax.bar(labels, vals, color='steelblue' if mt == 'BE' else 'darkorange', alpha=0.7)
            ax.set_title(pname, fontsize=9)
            ax.tick_params(labelsize=8)
            plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

        marker = ' \u2190 assigned' if mt == assigned else ''
        fig.suptitle(f'{aid} — {mt} phase-blocked params{marker}', fontsize=11)
        plt.tight_layout()
        plt.show()

### 5b. Per-Phase Model Assignment

Which model fits better at each learning phase? If the winner changes
across phases, the 3a expert-phase assignment may not apply to all phases.

This uses the in-sample error from the phase-blocked fit (not cross-validated),
which is sufficient for comparing models within the same phase.

In [9]:
# ── Per-phase model assignment ──────────────────────────────────────────────
phase_assignment_rows = []

for aid, fits in phase_fit_results.items():
    assigned = animal_phases[aid]['assigned_model']
    be_result = fits.get('BE', {})
    sc_result = fits.get('SC', {})

    if not be_result or not sc_result:
        continue

    # Get all phases present in both models
    all_phases = sorted(set(be_result.keys()) & set(sc_result.keys()))

    for phase in all_phases:
        be_err = be_result[phase].get('train_error', np.nan)
        sc_err = sc_result[phase].get('train_error', np.nan)

        if np.isnan(be_err) or np.isnan(sc_err):
            winner = '?'
        else:
            winner = 'BE' if be_err < sc_err else 'SC'

        phase_assignment_rows.append({
            'animal_id': aid,
            'phase': phase,
            'be_error': be_err,
            'sc_error': sc_err,
            'phase_winner': winner,
            'assigned_3a': assigned,
        })

pa_df = pd.DataFrame(phase_assignment_rows)

# ── Pivot table: animal × phase → winner ────────────────────────────────────
if not pa_df.empty:
    pivot = pa_df.pivot(index='animal_id', columns='phase', values='phase_winner')
    # Add 3a assignment column
    pivot['3a_assigned'] = pivot.index.map(
        lambda aid: animal_phases[aid]['assigned_model'])
    print('=== Model Assignment by Phase ===')
    print(pivot.to_string())

    # ── Flag animals where assignment changes across phases ───────────────────
    print('\n=== Assignment Stability ===')
    for aid in pivot.index:
        phase_winners = pivot.loc[aid].drop('3a_assigned').dropna().values
        unique = set(w for w in phase_winners if w != '?')
        assigned = pivot.loc[aid, '3a_assigned']

        if len(unique) == 1:
            model = unique.pop()
            match = '\u2713' if model == assigned or assigned not in ('BE', 'SC') else '\u2717'
            print(f'  {aid}: {model} in all phases (3a: {assigned}) {match}')
        elif len(unique) > 1:
            print(f'  {aid}: FLIPS across phases {dict(zip(pivot.columns[:-1], phase_winners))} '
                  f'(3a: {assigned}) \u26a0')
        else:
            print(f'  {aid}: no valid comparisons')

In [10]:
# ── Visual: BE vs SC error by phase ─────────────────────────────────────────
if not pa_df.empty:
    animals = pa_df['animal_id'].unique()
    n_animals = len(animals)
    fig, axes = plt.subplots(1, n_animals, figsize=(4 * n_animals, 4), sharey=True)
    if n_animals == 1:
        axes = [axes]

    for ax, aid in zip(axes, animals):
        sub = pa_df[pa_df['animal_id'] == aid].sort_values('phase')
        phases = sub['phase'].values
        x = np.arange(len(phases))
        w = 0.35

        ax.bar(x - w/2, sub['be_error'], w, color='steelblue', alpha=0.7, label='BE')
        ax.bar(x + w/2, sub['sc_error'], w, color='darkorange', alpha=0.7, label='SC')

        # Mark winner per phase
        for j, (_, row) in enumerate(sub.iterrows()):
            if row['phase_winner'] == 'BE':
                ax.plot(j - w/2, row['be_error'], 'v', color='black', ms=6)
            elif row['phase_winner'] == 'SC':
                ax.plot(j + w/2, row['sc_error'], 'v', color='black', ms=6)

        assigned = animal_phases[aid]['assigned_model']
        ax.set_title(f'{aid}\n(3a: {assigned})', fontsize=10)
        ax.set_xticks(x)
        ax.set_xticklabels(phases, fontsize=8, rotation=30, ha='right')
        if ax == axes[0]:
            ax.set_ylabel('UM MSE')
            ax.legend(fontsize=8)

    fig.suptitle('BE vs SC error by phase (\u25bc = winner)', fontsize=12)
    plt.tight_layout()
    plt.show()

## 6. Per-Session Fitting

The noisy but assumption-free parameter trajectory. Each session is
fit independently — expect high variance but the trend should be
visible if parameters are genuinely changing.

In [11]:
session_fit_results = {}  # aid -> {'BE': result, 'SC': result}

for aid, info in animal_phases.items():
    print(f'{aid}...', end=' ')
    animal_fits = {}
    for mt in ['BE', 'SC']:
        try:
            result = fit_sessions_individual(
                sessions=info['sessions'],
                model_type=mt,
                burn_in=BURN_IN,
                seed=42,
            )
            animal_fits[mt] = result
        except Exception as e:
            print(f'{mt} failed ({e})', end=' ')
            animal_fits[mt] = None
    session_fit_results[aid] = animal_fits
    print('done')

SS01... BE failed (too many values to unpack (expected 4)) SC failed (too many values to unpack (expected 4)) done
SS04... BE failed (too many values to unpack (expected 4)) SC failed (too many values to unpack (expected 4)) done
SS05... BE failed (too many values to unpack (expected 4)) SC failed (too many values to unpack (expected 4)) done
SS06... BE failed (too many values to unpack (expected 4)) SC failed (too many values to unpack (expected 4)) done
SS07... BE failed (too many values to unpack (expected 4)) SC failed (too many values to unpack (expected 4)) done
SS08... BE failed (too many values to unpack (expected 4)) SC failed (too many values to unpack (expected 4)) done
SS09... BE failed (too many values to unpack (expected 4)) SC failed (too many values to unpack (expected 4)) done
SS10... BE failed (too many values to unpack (expected 4)) SC failed (too many values to unpack (expected 4)) done
SS11... BE failed (too many values to unpack (expected 4)) SC failed (too many v

In [12]:
# ── Plot per-session parameter trajectories ─────────────────────────────────
for aid, fits in session_fit_results.items():
    assigned = animal_phases[aid]['assigned_model']

    for mt in ['BE', 'SC']:
        results = fits.get(mt)
        if results is None:
            continue

        param_names = BE_PARAMS if mt == 'BE' else SC_PARAMS
        # Extract per-session parameter values
        session_params = []
        for r in results:
            if r is not None and r.get('best_params'):
                session_params.append(r['best_params'])
            else:
                session_params.append({p: np.nan for p in param_names})

        if not session_params:
            continue

        n_params = len(param_names)
        fig, axes = plt.subplots(n_params, 1, figsize=(10, 2.5 * n_params), sharex=True)
        if n_params == 1:
            axes = [axes]

        colour = 'steelblue' if mt == 'BE' else 'darkorange'
        x = np.arange(len(session_params))

        for ax, pname in zip(axes, param_names):
            vals = [sp.get(pname, np.nan) for sp in session_params]
            ax.plot(x, vals, 'o-', ms=4, color=colour, alpha=0.7)
            ax.set_ylabel(pname, fontsize=9)
            ax.grid(True, alpha=0.2)

        axes[-1].set_xlabel('Session index')
        marker = ' \u2190 assigned' if mt == assigned else ''
        fig.suptitle(f'{aid} — {mt} per-session params{marker}', fontsize=11)
        plt.tight_layout()
        plt.show()

## 7. GP-Linked SBI Trajectory

The principled approach: fit all sessions jointly with a GP prior
on learning-rate parameters, enforcing smoothness while allowing
genuine changes.

**Requires:** Notebook 2c to validate GP-linked SBI on synthetic data first.

In [13]:
print('GP-linked SBI trajectory fitting: requires 2d validation first')
print('Placeholder — implement once SBIFitter is validated with GP priors')

GP-linked SBI trajectory fitting: requires 2d validation first
Placeholder — implement once SBIFitter is validated with GP priors


## 8. Summary

For each animal, summarise:
- Is the dynamic model better than static?
- Which parameters change across phases?
- Is the change consistent with the hypothesis (η high during learning,
  low during expert, high again post-shift)?

In [14]:
# ── Summary ──────────────────────────────────────────────────────────────────
for aid in sorted(comparison_results.keys()):
    comp = comparison_results[aid]
    assigned = animal_phases[aid]['assigned_model']
    print(f'\n{aid} (assigned: {assigned}):')

    for mt in ['BE', 'SC']:
        r = comp.get(mt)
        if r is None:
            continue
        marker = ' ← assigned' if mt == assigned else ''
        print(f'  {mt}: static={r["static_total_error"]:.4f} '
              f'dynamic={r["dynamic_total_error"]:.4f} '
              f'ratio={r["improvement_ratio"]:.3f}{marker}')

    # Check if assigned model agrees with best dynamic model
    dynamic_errors = {mt: comp[mt]['dynamic_total_error']
                      for mt in ['BE', 'SC'] if comp.get(mt)}
    if dynamic_errors:
        best_dynamic = min(dynamic_errors, key=dynamic_errors.get)
        if assigned in ('BE', 'SC') and best_dynamic != assigned:
            print(f'  ⚠ Dynamic best model ({best_dynamic}) differs from 3a assignment ({assigned})')


SS01 (assigned: Inconclusive):

SS04 (assigned: Inconclusive):

SS05 (assigned: None):

SS06 (assigned: None):

SS07 (assigned: None):

SS08 (assigned: None):

SS09 (assigned: None):

SS10 (assigned: None):

SS11 (assigned: None):

SS12 (assigned: None):

SS13 (assigned: None):
